# 01 · WHC A/B — uniform bucket vs SoilGrids/Saxton

**Question.** Does the per-pixel root-zone water-holding capacity derived from ISRIC SoilGrids 2.0 texture (Saxton–Rawls FC/WP, 250 m, `src/soil.py`) beat the uniform 100 mm fallback (`default_whc_mm`)? SoilGrids WHC was already the `run_cpi.py` default on physical grounds; this supplies — or withholds — the evidence.

**How to run:** put the `planting_pipeline` folder on your Google Drive, run top-to-bottom, approve the Drive-mount and Earth-Engine prompts. Export cells start GEE tasks and return immediately; the scoring cells read the CSVs once the tasks finish (watch https://code.earthengine.google.com/tasks).

## Setup

### Stage 0 · Runtime

Installs the Earth Engine client, geemap, pandas, geopandas and scipy. `scipy` is the one that matters
here: every score in this notebook is a leave-one-out cross-validation with a paired bootstrap interval,
and both come from scipy.

**Expected output.** `installed.`

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas scipy 2>/dev/null
print('installed.')

### Stage 0b · Earth Engine

`EE ready: ok`. The export cells below submit **batch tasks** and return immediately; the scoring cells
read the resulting CSVs. Between the two you have to wait, and you can close the browser while you do.
Watch the queue at code.earthengine.google.com/tasks.

**The export queue is per cloud project.** `ee-manzikye` has left batches in READY for hours. If the
tasks are not entering RUNNING within about 20 minutes, switch `PROJECT` to
`indigo-proxy-484220-q8` and resubmit rather than waiting.

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Drive

`pipeline on path: ...`. The scoring scripts read and write `Cropyield-Data/` inside this folder, so the
notebook must `chdir` here for the relative paths to resolve.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

### Scoring convention used throughout
`n` is small (6–81 zones) in every test here, and a single 70/30 split at n≈45 has a **±0.10 t/ha standard deviation — larger than any effect measured**. So every test below uses **leave-one-out CV** (each free parameter refit on n−1) plus a **paired bootstrap** CI, and reports **Spearman** alongside MAE because rank skill is invariant to the yield ceiling Ym. Reporting a single split would have produced two false positives in this round.

## 1. Export the arms
Identical pipeline in both arms; **only `whc_img` differs**. CPI is exported with `ym=1` so each arm can be given its own ceiling at scoring time.

> **Gotcha that invalidated the first run:** GAUL-2015 level 1 for Kenya is the **8 old provinces**, not the 47 counties HarvestStat uses. The first export returned 8 rows and would have matched 1 county. Everything now exports at **level 2** (old districts) and rolls up with `src/kenya_gaul_counties.py` (71 districts → 47 counties), **pixel-count weighted** so a 1-pixel district cannot outvote a 9,917-pixel one.

### Stage 1 · Export the two arms

**What runs.** Four export jobs. Both arms use the identical pipeline and **only `whc_img` differs**:
arm A is the uniform 100 mm bucket, arm B is the per-pixel SoilGrids and Saxton-Rawls root-zone
capacity.

| Script | Variant | Units |
|---|---|---|
| `whc_ab_test.py` | Kenya long rains, county | 44 |
| `whc_ab_variants.py --variant short_county` | Kenya short rains, county | 46 |
| `whc_ab_variants.py --variant short_ward` | Kenya short rains, 2021 and 2022 crop-cut wards | 66 and 15 |
| `whc_ab_variants.py --variant et_meher` | Ethiopia Meher, region | 6 |

**CPI is exported with `ym = 1`**, deliberately. That keeps the ceiling out of the export so each arm
can be given its own, or a shared one, at scoring time. Stage 3 is where that choice decides the answer.

**Expected output.** Task submission lines, returning in seconds. The exports themselves run for
minutes to hours.

**The gotcha that invalidated the first run, and why the exports are at level 2.** GAUL 2015 level 1
for Kenya is the **eight old provinces**, not the 47 counties HarvestStat reports. The first export
returned 8 rows and would have matched a single county. Everything now exports at level 2, the old
districts, and rolls up through `src/kenya_gaul_counties.py`, 71 districts to 47 counties,
**weighted by pixel count** so a one-pixel district cannot outvote a 9,917-pixel one.

In [ ]:
!python whc_ab_test.py            # Kenya Long rains, county
!python whc_ab_variants.py --variant short_county
!python whc_ab_variants.py --variant short_ward     # 2021 + 2022 crop-cut wards
!python whc_ab_variants.py --variant et_meher

## 2. Score — per-arm Ym (the original protocol)

### Stage 2 · Score with a per-arm ceiling

**What runs.** Each arm is given **its own** fitted $Y_m$, then scored by leave-one-out MAE and
Spearman against HarvestStat.

**Why leave-one-out rather than a 70/30 split.** With 6 to 81 zones, a single 70/30 split at $n\approx45$
has a standard deviation of about **±0.10 t/ha**, which is larger than any effect measured in this whole
round of tests. A single split would have produced **two false positives** here. Every free parameter is
refit on $n-1$, and the interval is a paired bootstrap.

**Expected output.** A per-variant table of $\Delta$ MAE with a confidence interval. Read the interval,
not the point estimate: only one variant produces an interval clear of zero, and stage 3 removes it.

In [ ]:
!python whc_ab_score.py
!python whc_ab_score_variants.py --variant short_ward
!python whc_ab_score_variants.py --variant short_county
!python whc_ab_score_variants.py --variant et_meher

## 3. Score — Ym HELD FIXED across arms
**This is the step that changes the answer.** SoilGrids WHC is uniformly a *bigger* bucket, so its dominant effect is a **level shift** in CPI — and refitting a separate Ym per arm absorbs exactly that by construction. Three shared-ceiling regimes: `prod` (the ceiling `src/cpi.py` actually ships), `fit_A` and `fit_B` (fitted on one arm, applied to both) — the latter two bracket the answer so a result cannot be an artifact of normalisation.

### Stage 3 · Score with the ceiling held fixed, which is the step that changes the answer

**What runs.** The same comparison with **one shared $Y_m$ across both arms**, under three regimes:
`prod`, the ceiling the pipeline ships; `fit_A`; and `fit_B`. The last two bracket the answer, so a
result cannot be an artefact of which arm set the level.

**Why it matters.** SoilGrids WHC is uniformly a **bigger bucket** than 100 mm, so its dominant effect
on CPI is a **level shift**. Refitting a separate $Y_m$ per arm absorbs exactly that shift by
construction, which is how a level difference disguises itself as a skill difference.

**Expected values.**

| Variant | n | Per-arm Ym Δ(A−B) | Fixed Ym Δ(A−B) | Verdict |
|---|---|---|---|---|
| Kenya long, county | 44 | +0.010 [−0.060, +0.071] | −0.024 [−0.076, +0.023] | null |
| Kenya short, county | 46 | +0.046 [+0.006, +0.085] | +0.010 [−0.049, +0.066] | **the win does not survive** |
| Kenya short, ward 2021 | 66 | −0.000 | −0.082 [−0.101, −0.064] | uniform better |
| Kenya short, ward 2022 Kitui | 15 | +0.006 | **−0.318 [−0.384, −0.240]** | uniform much better |
| Ethiopia Meher, region | 6 | +0.002 | −0.006 | arms identical |

**How to read this.** The single positive result was an artefact of the per-arm ceiling. Under a shared
ceiling SoilGrids is never better and is clearly worse in Kitui 2022, where the bigger bucket **hides
the drought**: it carries enough stored water through the dry spell for the model not to register it.
Ethiopia's arms are identical because Meher WRSI saturates near 100 in both.

**What this does and does not license.** SoilGrids WHC remains defensible on physical grounds and stays
the default. What cannot be claimed is that it improves yield skill; the evidence says it does not.
A null result recorded properly is the point of the exercise.

In [ ]:
!python whc_ab_score_ymfixed.py

## Result

| variant | n | per-arm Ym Δ(A−B) | **fixed Ym Δ(A−B)** | verdict |
|---|---|---|---|---|
| KE Long, county | 44 | +0.010 [−0.060,+0.071] | −0.024 [−0.076,+0.023] | null |
| KE Short, county | 46 | **+0.046 [+0.006,+0.085]** | +0.010 [−0.049,+0.066] | **win does NOT survive** |
| KE Short, ward 2021 | 66 | −0.000 | −0.082 [−0.101,−0.064] | **uniform better** |
| KE Short, ward 2022 Kitui | 15 | +0.006 | **−0.318 [−0.384,−0.240]** | **uniform much better** |
| ET Meher, region | 6 | +0.002 | −0.006 | arms identical |

**1. The one positive result was an artifact.** The short-rains county win vanished once both arms shared a ceiling — it was arm B being allowed its own better-fitting Ym.

**2. SoilGrids improves spatial ranking nowhere**, in any variant, under any regime.

**3. In the one true failure season it HIDES the drought.** Kitui SR-2022, all 15 wards under 0.15 t/ha: the uniform bucket flags **14/15** wards as severe (CPI<25); SoilGrids flags **5/15**. For early warning the uniform bucket's pessimism is a feature.

**4. Ethiopia proves when the bucket cannot matter at all.** Flowering WRSI is 99.5–100 in every region under both arms — the Meher balance is fully satisfied, the store never draws down, so bucket size is arithmetically irrelevant. Don't re-run this test for Meher.

**Recommendation.** Keep SoilGrids/Saxton as default on physical-realism grounds (per-pixel TAW is the defensible physics and costs nothing), but state in the report that it is **not skill-validated**, and do not cite the short-rains MAE as supporting evidence.